# 03. Model Training & Validation Benchmarking
**Course Project:** Cybercrime Complaint Classification Using Machine Learning  
**Objective:** Train and compare three candidate NLP pipelines:
1. **TF-IDF + Linear SVM (Softmax Decision Calibration)**
2. **TF-IDF + Logistic Regression**
3. **TF-IDF + Random Forest**

### Optimizations for Unseen Real-World Generalization:
- **Balanced Training Strategy**: Neutralizes the 12:1 Cyberbullying imbalance by equalizing class representations during training (~350 samples each).
- **Platform-Neutral Stop Words**: Removes social media tokens (`'twitter'`, `'tweet'`) from stop words to prevent spurious correlations.
- **Sublinear TF Scaling**: Dampens high-frequency word dominance.
- **Zero Test-Set Exposure**: Validation data is strictly for model selection; final test sets remain untouched until evaluation.

In [ ]:
import os
import sys
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Ensure src directory is accessible
sys.path.insert(0, os.path.abspath('../src'))
from text_pipeline import build_tfidf_vectorizer, CANONICAL_CATEGORIES
from train_models import SoftmaxLinearSVC

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

# Load datasets
train_df = pd.read_csv('../data/processed/train.csv')
val_df = pd.read_csv('../data/processed/validation.csv')

# Balanced training sampling: equalize Cyberbullying to match the other categories
RANDOM_SEED = 42
cb_mask = train_df['category'] == 'Cyberbullying / Online Harassment'
target_count = int(train_df[~cb_mask]['category'].value_counts().median())
cb_sample = train_df[cb_mask].sample(target_count, random_state=RANDOM_SEED)
balanced_train_df = pd.concat([train_df[~cb_mask], cb_sample]).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

X_train, y_train = balanced_train_df['complaint_text'], balanced_train_df['category']
X_val, y_val = val_df['complaint_text'], val_df['category']

print(f"Raw Training records:      {len(train_df):,}")
print(f"Balanced Training records: {len(X_train):,} ({target_count} per class)")
print(f"Validation records:        {len(X_val):,}")

## 1. Candidate Pipelines Setup

In [ ]:
pipelines = {
    'Linear_SVM': Pipeline([
        ('tfidf', build_tfidf_vectorizer(min_df=1, max_df=0.85, ngram_range=(1, 2), sublinear_tf=True)),
        ('clf', SoftmaxLinearSVC(C=1.0, random_state=RANDOM_SEED, dual=False, max_iter=3000))
    ]),
    'Logistic_Regression': Pipeline([
        ('tfidf', build_tfidf_vectorizer(min_df=1, max_df=0.85, ngram_range=(1, 2), sublinear_tf=True)),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0, random_state=RANDOM_SEED))
    ]),
    'Random_Forest': Pipeline([
        ('tfidf', build_tfidf_vectorizer(min_df=1, max_df=0.85, ngram_range=(1, 2), sublinear_tf=True)),
        ('clf', RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1))
    ])
}
print("Pipelines configured:", list(pipelines.keys()))

## 2. Train and Validate on Validation Partition

In [ ]:
results = []
val_predictions = {'id': val_df['id'], 'true_category': y_val}

for name, pipe in pipelines.items():
    print(f"\nTraining {name}...")
    t0 = time.time()
    pipe.fit(X_train, y_train)
    duration = time.time() - t0
    
    preds = pipe.predict(X_val)
    val_predictions[f'pred_{name}'] = preds
    
    acc = accuracy_score(y_val, preds)
    macro_prec = precision_score(y_val, preds, average='macro', zero_division=0)
    macro_rec = recall_score(y_val, preds, average='macro', zero_division=0)
    macro_f1 = f1_score(y_val, preds, average='macro', zero_division=0)
    weighted_f1 = f1_score(y_val, preds, average='weighted', zero_division=0)
    
    results.append({
        'Model': name.replace('_', ' '),
        'Training Time (s)': round(duration, 2),
        'Validation Accuracy': round(acc, 4),
        'Macro Precision': round(macro_prec, 4),
        'Macro Recall': round(macro_rec, 4),
        'Macro F1-Score': round(macro_f1, 4),
        'Weighted F1-Score': round(weighted_f1, 4)
    })
    print(f"Done in {duration:.2f}s | Val Acc: {acc*100:.2f}% | Macro F1: {macro_f1*100:.2f}%")

comparison_df = pd.DataFrame(results).sort_values(by='Macro F1-Score', ascending=False)
comparison_df

## 3. Confusion Matrix Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

for i, (name, pipe) in enumerate(pipelines.items()):
    preds = val_predictions[f'pred_{name}']
    cm = confusion_matrix(y_val, preds, labels=CANONICAL_CATEGORIES)
    short_labels = [l[:12] + '...' if len(l) > 12 else l for l in CANONICAL_CATEGORIES]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=short_labels, yticklabels=short_labels, cbar=False)
    axes[i].set_title(f"{name.replace('_', ' ')}", fontweight='bold')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('True' if i == 0 else '')
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle('Validation Confusion Matrices across Candidate Pipelines', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Testing Inference on Unseen Custom Complaints

In [ ]:
sample_prompts = [
    "I received a message asking me to verify my banking password on a fake portal.",
    "Someone created a fake Instagram profile with my stolen photos to scam my friends.",
    "My files were locked by ransomware and they are demanding 0.5 Bitcoin to decrypt.",
    "An unauthorized charge of $1,200 was made on my debit card without my knowledge."
]

best_pipe = pipelines['Linear_SVM']
preds = best_pipe.predict(sample_prompts)
probs = best_pipe.predict_proba(sample_prompts)

for text, pred, prob in zip(sample_prompts, preds, probs):
    conf = np.max(prob) * 100
    print(f"Text: \"{text}\"")
    print(f" --> Predicted: [{pred}] (Confidence: {conf:.1f}%)\n")